# Gradio Web Application for Full Fine-Tuned Model
## Objective
The objective of this notebook is to test and launch a live, interactive web application for the Fully Fine-Tuned NLLB translation model. It contains the complete, self-contained Python code required to build a Gradio-based user interface that allows users to perform bidirectional Odia-German translation.

## Methodology
The script is designed to be a complete web application.

1. **Model Loading:** It loads the final, fully fine-tuned model and its corresponding tokenizer directly from their repository on the Hugging Face Hub.
2. **Language Detection:** It implements a robust, hybrid language detection system. It first uses a script-based check to reliably identify Odia text. If the text is not Odia, it uses the `langdetect` library as a fallback to identify German or other languages.
3. **Translation Logic:** It defines a central `translate_text` function that takes user input, runs the detection logic, and calls the loaded `pipeline` with the correct source and target language codes to generate the translation.
4. **Web Interface:** It uses the `gradio` library to create a clean and intuitive user interface, complete with text boxes, a dropdown for manual language selection, and example sentences.

## Workflow
1. Installs all required libraries (`gradio`, `transformers`, `langdetect`, etc.).
2. Loads the fully fine-tuned model and tokenizer from the Hub.
3. Defines the language detection and translation functions.
4. Creates the Gradio `Interface` object.
5. Launches the web application, creating a temporary public URL for testing in the Colab environment.

## Input & Output
* **Input:** Text entered by a user into the Gradio web interface.
* **Output:** A live, interactive Gradio web application for bidirectional Odia-German translation.

In [1]:
# Uninstall all potentially conflicting packages
!pip uninstall -y torch torchvision torchaudio transformers gradio langdetect sentencepiece accelerate huggingface-hub safetensors peft torchtune sentence-transformers timm bitsandbytes

Found existing installation: torch 2.9.0
Uninstalling torch-2.9.0:
  Successfully uninstalled torch-2.9.0
Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
Found existing installation: gradio 5.50.0
Uninstalling gradio-5.50.0:
  Successfully uninstalled gradio-5.50.0
Found existing installation: langdetect 1.0.9
Uninstalling langdetect-1.0.9:
  Successfully uninstalled langdetect-1.0.9
Found existing installation: sentencepiece 0.2.1
Uninstalling sentencepiece-0.2.1:
  Successfully uninstalled sentencepiece-0.2.1
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: huggingface-hub 0.36.0
Uninstalling huggingface-hub-0.36.0:
  Successfully uninstalled huggingface-hub-0.36.0
Found existing installation: safetensors 0.7.0
Uninstalling safetensors-0.7.0:
  Successfully uninstalled safetensors-0.7.0


In [2]:
# Clear pip cache
!pip cache purge

Files removed: 53


In [3]:
# Install libraries
!pip install torch==2.9.0
!pip install transformers==4.57.3
!pip install gradio==5.50.0
!pip install langdetect==1.0.9
!pip install sentencepiece==0.2.1
!pip install huggingface-hub==0.36.0
!pip install accelerate==1.12.0
!pip install safetensors==0.7.0
!pip install bitsandbytes==0.49.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.6 requires torchvision>=0.11, which is not installed.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 38.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 61.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=ecb3fffefbc7a1b0018d4ba944bc9f2d3f770a42696a3f2eb87a2933cf2edae

In [4]:
# Verify installations
!pip show torch transformers gradio langdetect sentencepiece huggingface-hub accelerate safetensors bitsandbytes

Name: torch
Version: 2.9.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvshmem-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, bitsandbytes, fastai, torchdata
---
Name: transformers
Version: 4.57.3
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (htt

In [5]:
# Check CUDA availability and version
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")

CUDA Available: True
CUDA Version: 12.8


In [6]:
# clear cache
!rm -rf ~/.cache/huggingface

In [7]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")

PyTorch Version: 2.9.0+cu128
CUDA Available: True
CUDA Version: 12.8


In [8]:
# 1. Import libraries
import torch
import gradio as gr
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig
from langdetect import detect, LangDetectException
import re
import logging
import traceback
from huggingface_hub import login
from google.colab import userdata

In [9]:
# Set up logging for tracking translations
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [10]:
# Authenticate with Hugging Face Hub
huggingface_token = userdata.get('HF_TOKEN')
login(token=huggingface_token)

In [11]:
# 2. Configuration for FFT Model
# This is the standalone model where all 600M parameters were updated
FFT_MODEL_HUB_ID = "abhinandansamal/nllb-200-distilled-600M-full-finetuned-odia-german-bidirectional"
ODIA_LANG_CODE = "ory_Orya"
GERMAN_LANG_CODE = "deu_Latn"

# Task Prefixes used during the FFT training phase
PREFIX_ORI_TO_DEU = "translate Odia to German: "
PREFIX_DEU_TO_ORI = "translate German to Odia: "

In [12]:
# 3. Model Loading (Fixed for FFT Logic)
print("🚀 Loading the Full Fine-Tuned (FFT) Model...")

try:
    # 8-bit Quantization: Matches your final evaluation setup for FFT
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)

    tokenizer = AutoTokenizer.from_pretrained(FFT_MODEL_HUB_ID)

    # We load the full weight model (FFT) directly
    model = AutoModelForSeq2SeqLM.from_pretrained(
        FFT_MODEL_HUB_ID,
        quantization_config=bnb_config,
        device_map="auto"
    )

    # Creating the pipeline with a generation limit of 512 (Standard NLLB sync)
    # Using 'num_beams=5' to match "Standard Inference" best performer
    translator = pipeline(
        "translation",
        model=model,
        tokenizer=tokenizer,
        max_length=512,
        num_beams=5,
        length_penalty=1.0
    )
    print("✅ FFT Model and pipeline loaded successfully in 8-bit!")
except Exception as e:
    print(f"❌ Initialization Error: {traceback.format_exc()}")
    translator = None

🚀 Loading the Full Fine-Tuned (FFT) Model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/836 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

Device set to use cuda:0


✅ FFT Model and pipeline loaded successfully in 8-bit!


In [13]:
# 4. Helper Functions
def is_odia_script(text):
    """
    Checks if the input text contains characters from the Odia Unicode block.

    This function scans the string for characters in the range U+0B00 to U+0B7F,
    which corresponds to the Odia script. It is used as a heuristic for language detection.

    Args:
        text (str): The input text to analyze.

    Returns:
        bool: True if at least one Odia character is found, False otherwise.
    """
    if not text: return False

    # Compile regex pattern for the Odia Unicode block (U+0B00–U+0B7F)
    odia_pattern = re.compile(r'[\u0B00-\u0B7F]')

    # .search() returns a match object if found, None otherwise
    return bool(odia_pattern.search(text))

def translate_logic(input_text, source_lang="auto"):
    """
    Core translation logic handling language detection, prefixing, and inference.

    This function orchestrates the translation process for the Full Fine-Tuned (FFT) model.
    It automatically routes the request based on the detected or specified source language,
    applies the correct NLLB task prefix, and invokes the translation pipeline.

    Args:
        input_text (str): The sentence to be translated.
        source_lang (str, optional): The language code ('auto', 'or', or 'de').
                                     Defaults to "auto".

    Returns:
        str: The translated text or an error message if the process fails.
    """
    # Basic validation
    if translator is None: return "Error: Model not loaded."
    if not input_text.strip(): return "Error: Please enter text."

    try:
        # --- 1. Language Detection Strategy ---
        if source_lang == "auto":
            # Primary Heuristic: Check specifically for Odia script first
            if is_odia_script(input_text):
                lang = "or"
            else:
                # Fallback: General-purpose language detection library (e.g., langdetect)
                try:
                    lang = detect(input_text)
                except:
                    return "Error: Language detection failed."
        else:
            # User manually forced a specific language
            lang = source_lang

        # --- 2. Prefix Application & Routing ---
        # The NLLB model requires specific prefixes to activate the correct translation head.
        if lang == "or":
            prompt = PREFIX_ORI_TO_DEU + input_text
            src, tgt = ODIA_LANG_CODE, GERMAN_LANG_CODE
            logger.info("Executing FFT Translation: Odia -> German")
        elif lang == "de":
            prompt = PREFIX_DEU_TO_ORI + input_text
            src, tgt = GERMAN_LANG_CODE, ODIA_LANG_CODE
            logger.info("Executing FFT Translation: German -> Odia")
        else:
            # Fail gracefully for unsupported languages
            return f"Error: Language '{lang}' not supported."

        # --- 3. Perform Inference ---
        # The 'translator' pipeline object handles tokenization, generation, and decoding.
        res = translator(prompt, src_lang=src, tgt_lang=tgt)

        # Extract the translation string from the pipeline output list/dict
        return res[0]["translation_text"]

    except Exception as e:
        logger.error(f"FFT Translation Failure: {e}")
        return f"Error: {str(e)}"

In [14]:
# ==========================================
# 5. UI CONSTRUCTION (GRADIO)
# ==========================================

# UI Metadata
title = "💎 Full Fine-Tuned Odia-German Translator"
description = """
### Full Weight Update (FFT)
This application uses the **Fully Fine-Tuned NLLB-200 (600M)** model.
Unlike the LoRA version, this model has had all its internal weights optimized for this language pair.
"""

# Pre-defined examples to help users test the model quickly
examples = [
    ["ଆଜି ପାଗ ବହୁତ ଭଲ ଅଛି।", "or"],     # "The weather is very good today."
    ["Wie ist deine Gesundheit?", "de"],      # "How is your health?"
    ["ମନ୍ତ୍ରୀ ଘୋଷଣା କଲେ ଯେ ଏହି ନୂଆ ରାଜପଥ ଆସନ୍ତା ବର୍ଷ ସୁଦ୍ଷା ସମ୍ପୂର୍ଣ୍ଣ ହେବ।", "or"]      # Complex political sentence
]

# Initialize Gradio Interface
iface = gr.Interface(
    fn=translate_logic,
    inputs=[
        gr.Textbox(lines=4, label="Input Sentence", placeholder="Type here..."),
        gr.Radio(choices=["auto", "or", "de"], label="Source Language", value="auto")
    ],
    outputs=gr.Textbox(lines=4, label="FFT Translation Output"),
    title=title,
    description=description,
    examples=examples,
    theme=gr.themes.Monochrome()
)

# --- LAUNCH APPLICATION ---
if __name__ == "__main__":
    # share=True creates a public link (valid for 72h) for external access
    iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b0250f9c354adf8b70.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
